# ERA V5 · Session 13 — reversibility, measured

A reversible stack throws its activations away in the forward pass and rebuilds them during
the backward pass, so activation memory stops growing with depth. This notebook builds the
three variants from the paper (arXiv:2512.02056) by hand, checks that their gradients match
ordinary autograd, measures what the memory buys, and trains the three runs the assignment asks
for at reduced length.

Everything runs live. A GPU helps; the correctness checks run anywhere.

In [1]:
import os, sys, json
if not os.path.exists('train.py'):              # on Colab: fetch the code
    !git clone -q https://github.com/sandeepkunkunuru/era-v5.git 2>/dev/null || true
    os.chdir('era-v5/session-13-reversibility')
sys.path.insert(0, os.getcwd())
import torch, train as T, model as M
print('torch', torch.__version__, '| device', T.DEV)

torch 2.12.1+cu130 | device cuda


## The four stacks

Only the rule carrying the hidden state between layers changes. `f` is the whole transformer
block in every case (the paper's eq. 2.5).

| stack | rule | reversible |
|---|---|---|
| baseline | `p[l+1] = p[l] + f(p[l])` | no — stores every layer's input |
| midpoint | `p[l+1] = a·p[l-1] + 2h·f(p[l])` | yes |
| leapfrog | `p[l+1] = 2p[l] - p[l-1] + h²·f(p[l])` | yes |
| Hamiltonian | `q[l] = a·q[l-1] + Attn(LN(p[l-1]))`, `p[l] = a·p[l-1] + MLP(LN(q[l]))` | yes |

Plain forward Euler is **not** reversible: undoing `p[l+1] = p[l] + h·f(p[l])` needs `f` at the
state you are trying to recover.

## 1. Do the rebuilt gradients match ordinary autograd?

The custom backward reconstructs each layer's input instead of remembering it. If that is right,
the parameter gradients must equal what plain autograd produces on the same maths. In fp64:

In [2]:
for r in T.grad_check():
    print(f"{r['stack']:12s} worst relative gradient error {r['worst_rel_grad_error']:.2e}")

midpoint     worst relative gradient error 8.22e-16
leapfrog     worst relative gradient error 6.61e-16
hamiltonian  worst relative gradient error 1.21e-15


## 2. Reconstruction, and what damping does to it

V4's production integrator ran `a = 0.5`. The paper's stability analysis requires `|a| = 1` for a
method to be stable forwards *and* backwards. The midpoint inverse divides by `a` at every layer,
so a damped `a` multiplies any error on the way down.

In [3]:
for r in T.reconstruction_table():
    print(f"{r['stack']:12s} a={r['a']}  relative error {r['rel_error']:.3e}")

midpoint     a=1.0  relative error 2.985e-14
midpoint     a=0.5  relative error 2.973e-11
leapfrog     a=1.0  relative error 6.984e-15
leapfrog     a=0.5  relative error 6.984e-15
hamiltonian  a=1.0  relative error 1.083e-13
hamiltonian  a=0.5  relative error 1.364e-06


## 3. How much batch does the memory buy?

The largest batch that survives two full steps, found by doubling then bisection.

In [4]:
caps = {s: T.max_batch(s) for s in ('baseline', 'midpoint')}
print(caps, '→', f"{caps['midpoint'] / caps['baseline']:.1f}x")

{'baseline': 26, 'midpoint': 179} → 6.9x


## 4. The three runs, shortened

Baseline at a fixed batch, reversible at the same batch, reversible at its own maximum.

In [5]:
TOK = 400_000        # the full assignment uses 50M
b = caps['baseline']
rows = [T.run('baseline', b, TOK, '1-baseline'),
        T.run('midpoint', b, TOK, '2-reversible-same-batch'),
        T.run('midpoint', caps['midpoint'], TOK, '3-reversible-max-batch')]
print()
print(f"{'run':26s} {'batch':>6s} {'tok/s':>10s} {'peak MiB':>10s} {'val loss':>9s}")
for r in rows:
    print(f"{r['tag']:26s} {r['batch']:6d} {r['tokens_per_s']:10,.0f} "
          f"{r['peak_mem_mib']:10,.0f} {r['val_loss']:9.4f}")

  [1-baseline] step 0/30 loss 5.5363


  [1-baseline] done: val 3.3496 · 22,566 tok/s · 5,243 MiB peak


  [2-reversible-same-batch] step 0/30 loss 5.5186


  [2-reversible-same-batch] done: val 3.3844 · 14,097 tok/s · 1,069 MiB peak


  [3-reversible-max-batch] step 0/4 loss 5.5073


  [3-reversible-max-batch] out of memory -- backing off to batch 152


  [3-reversible-max-batch] step 0/5 loss 5.5099


  [3-reversible-max-batch] done: val 4.0470 · 14,026 tok/s · 4,537 MiB peak



run                         batch      tok/s   peak MiB  val loss
1-baseline                     26     22,566      5,243    3.3496
2-reversible-same-batch        26     14,097      1,069    3.3844
3-reversible-max-batch        152     14,026      4,537    4.0470


## 5. Memory against depth

The structural claim: the baseline carries a per-layer activation term and the reversible stack
does not, so the gap widens as the stack gets deeper.

In [6]:
for r in T.memory_vs_depth(max(1, caps['baseline'] // 2), [4, 8, 12, 24]):
    print(f"{r['layers']:3d} layers  {r['stack']:9s} {r['peak_mem_mib']:8,.0f} MiB")

    4 layers  baseline         922 MiB


    4 layers  midpoint         438 MiB


    8 layers  baseline       1,756 MiB


    8 layers  midpoint         492 MiB


   12 layers  baseline       2,590 MiB


   12 layers  midpoint         546 MiB


   24 layers  baseline       5,091 MiB


   24 layers  midpoint         841 MiB


  4 layers  baseline       922 MiB
  4 layers  midpoint       438 MiB
  8 layers  baseline     1,756 MiB
  8 layers  midpoint       492 MiB
 12 layers  baseline     2,590 MiB
 12 layers  midpoint       546 MiB
 24 layers  baseline     5,091 MiB
 24 layers  midpoint       841 MiB


## What to take away

- The reversible stacks reproduce ordinary autograd's gradients to floating-point noise, so the
  memory saving costs nothing in correctness.
- It is not free in time: rebuilding the activations re-runs every block, which the paper prices
  at 30–50% and these runs confirm.
- It only pays when the recovered memory is spent — on a bigger batch, a longer sequence, or a
  deeper stack.
- Dropout must be zero, or the rebuild draws a different mask and the gradients are wrong.